In [1]:
import spark_config
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LogAnalyzer") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark {spark.version} ready!")

Spark 3.5.7 ready!


In [3]:
import os

BASE_DIR = os.path.dirname(os.getcwd()) if "notebooks" in os.getcwd() else os.getcwd()
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

csv_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".csv")]
print(f"Found: {csv_files}")

file_path = os.path.join(RAW_DATA_DIR, csv_files[0])
df = spark.read.csv(file_path, header=True, inferSchema=True)

print(f"Rows: {df.count():,}")
print(f"Columns: {df.columns}")
df.show(5)

Found: ['labeled.csv']
Rows: 9,282,184
Columns: ['ip', 'time', 'method', 'url', 'protocol', 'status', 'size', 'referrer', 'user_agent', 'extra', 'no', 'label', 'type']
+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+-----+---+-----+------+
|           ip|               time|method|                 url|protocol|status| size|            referrer|          user_agent|extra| no|label|  type|
+-------------+-------------------+------+--------------------+--------+------+-----+--------------------+--------------------+-----+---+-----+------+
|  31.56.96.51|2019-01-22 05:56:16|   GET|/image/60844/prod...|HTTP/1.1|   200| 5667|https://www.zanbi...|Mozilla/5.0 (Linu...|    -|  2|    0|benign|
|  31.56.96.51|2019-01-22 05:56:16|   GET|/image/61474/prod...|HTTP/1.1|   200| 5379|https://www.zanbi...|Mozilla/5.0 (Linu...|    -|  3|    0|benign|
|  91.99.72.15|2019-01-22 05:56:17|   GET|/product/31893/62...|HTTP/1.1|   20

In [ ]:
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

In [ ]:
from pyspark.sql.functions import col

# Valid values we identified from our exploration
valid_protocols = ["HTTP/1.1", "HTTP/1.0"]
valid_types = ["benign", "bot", "sqli", "scanning", "rce"]
valid_labels = ["0", "1"]

# Keep only rows where these columns have valid values
df_clean = df.filter(
    col("protocol").isin(valid_protocols) &
    col("type").isin(valid_types) &
    col("label").isin(valid_labels)
)

original_count = df.count()
clean_count = df_clean.count()
dropped = original_count - clean_count

print(f"Original rows:  {original_count:,}")
print(f"Clean rows:     {clean_count:,}")
print(f"Dropped rows:   {dropped:,} ({(dropped/original_count)*100:.2f}%)")

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, LongType

# Drop the 'extra' column (99.35% empty, useless)
# Cast columns to proper types
df_clean = df_clean \
    .drop("extra") \
    .withColumn("label", col("label").cast(IntegerType())) \
    .withColumn("no", col("no").cast(LongType())) \
    .withColumn("size", col("size").cast(LongType())) \
    .withColumn("status", col("status").cast(IntegerType()))

print("Updated schema:")
df_clean.printSchema()
df_clean.show(5)

In [ ]:
import os

CLEAN_DATA_DIR = os.path.join(BASE_DIR, "data", "clean")
os.makedirs(CLEAN_DATA_DIR, exist_ok=True)

# Save as parquet (much faster and smaller than CSV for Spark)
df_clean.write.mode("overwrite").parquet(os.path.join(CLEAN_DATA_DIR, "logs_clean.parquet"))

print(f"Saved clean data to: {CLEAN_DATA_DIR}")
print(f"Clean rows: {df_clean.count():,}")

In [4]:
# Load clean data (use this going forward instead of re-cleaning)
import os
CLEAN_DATA_DIR = os.path.join(BASE_DIR, "data", "clean")
df_clean = spark.read.parquet(os.path.join(CLEAN_DATA_DIR, "logs_clean.parquet"))
print(f"Loaded {df_clean.count():,} clean rows")

Loaded 9,245,740 clean rows


In [5]:
from pyspark.sql.functions import count, col, round as spark_round

# Total requests
total = df_clean.count()

# Attack vs Benign breakdown
type_breakdown = df_clean.groupBy("type") \
    .agg(
        count("*").alias("count")
    ) \
    .withColumn("percentage", spark_round((col("count") / total) * 100, 2)) \
    .orderBy("count", ascending=False)

print(f"=== SUMMARY ===")
print(f"Total requests: {total:,}")
print(f"\nBreakdown by type:")
type_breakdown.show(truncate=False)

=== SUMMARY ===
Total requests: 9,245,740

Breakdown by type:
+--------+-------+----------+
|type    |count  |percentage|
+--------+-------+----------+
|benign  |9242998|99.97     |
|bot     |1526   |0.02      |
|sqli    |718    |0.01      |
|scanning|474    |0.01      |
|rce     |24     |0.0       |
+--------+-------+----------+



In [6]:
from pyspark.sql.functions import count, col, collect_set

# Filter only attack traffic (non-benign)
df_attacks = df_clean.filter(col("type") != "benign")

print(f"Total attack requests: {df_attacks.count():,}")

# Top 20 attacker IPs with what types of attacks they performed
top_attackers = df_attacks.groupBy("ip") \
    .agg(
        count("*").alias("attack_count"),
        collect_set("type").alias("attack_types")
    ) \
    .orderBy("attack_count", ascending=False)

print("\n=== TOP 20 ATTACKER IPs ===")
top_attackers.show(20, truncate=False)

Total attack requests: 2,742

=== TOP 20 ATTACKER IPs ===
+---------------+------------+------------+
|ip             |attack_count|attack_types|
+---------------+------------+------------+
|5.101.40.234   |668         |[sqli]      |
|216.244.66.248 |310         |[bot]       |
|176.121.14.183 |50          |[sqli]      |
|78.217.202.91  |50          |[scanning]  |
|54.209.60.63   |48          |[bot]       |
|184.72.115.35  |40          |[bot]       |
|18.223.239.161 |32          |[bot]       |
|92.87.91.22    |32          |[scanning]  |
|91.242.162.85  |29          |[bot]       |
|54.175.74.27   |26          |[bot]       |
|162.210.196.97 |24          |[bot]       |
|199.58.86.211  |23          |[bot]       |
|162.210.196.129|22          |[bot]       |
|54.86.66.252   |21          |[bot]       |
|123.206.9.252  |20          |[scanning]  |
|108.59.8.70    |20          |[bot]       |
|162.210.196.98 |20          |[bot]       |
|190.85.81.27   |20          |[scanning]  |
|199.58.86.209  |2

In [7]:
# Which URLs are being attacked the most and by what type
targeted_urls = df_attacks.groupBy("url", "type") \
    .agg(count("*").alias("attack_count")) \
    .orderBy("attack_count", ascending=False)

print("=== TOP 20 TARGETED URLs ===")
targeted_urls.show(20, truncate=False)

=== TOP 20 TARGETED URLs ===
+--------------------------------------------------------------------------------------------------------------------------------------+--------+------------+
|url                                                                                                                                   |type    |attack_count|
+--------------------------------------------------------------------------------------------------------------------------------------+--------+------------+
|/robots.txt                                                                                                                           |bot     |1524        |
|/wp-login.php                                                                                                                         |scanning|250         |
|/test.php                                                                                                                             |scanning|12          |
|/login.cgi?cli=a

In [8]:
from pyspark.sql.functions import count, col

# Requests with no referrer (direct access) that are attacks
direct_attacks = df_attacks.filter(
    (col("referrer").isNull()) | (col("referrer") == "-")
)

direct_vs_referred = df_attacks.groupBy(
    (col("referrer").isNull() | (col("referrer") == "-")).alias("is_direct")
).agg(count("*").alias("count"))

print("=== ATTACK TRAFFIC: DIRECT vs REFERRED ===")
direct_vs_referred.show(truncate=False)

# What are direct-access attacks hitting?
print("\n=== TOP DIRECT-ACCESS ATTACK URLs ===")
direct_attacks.groupBy("url", "type") \
    .agg(count("*").alias("count")) \
    .orderBy("count", ascending=False) \
    .show(15, truncate=False)

=== ATTACK TRAFFIC: DIRECT vs REFERRED ===
+---------+-----+
|is_direct|count|
+---------+-----+
|true     |2702 |
|false    |40   |
+---------+-----+


=== TOP DIRECT-ACCESS ATTACK URLs ===
+--------------------------------------------------------------------------------------------------------------------------------------+--------+-----+
|url                                                                                                                                   |type    |count|
+--------------------------------------------------------------------------------------------------------------------------------------+--------+-----+
|/robots.txt                                                                                                                           |bot     |1499 |
|/wp-login.php                                                                                                                         |scanning|243  |
|/test.php                                       

In [9]:
from pyspark.sql.functions import count, col, min as spark_min, max as spark_max, round as spark_round
from pyspark.sql.functions import unix_timestamp

# For each IP: total requests, time span, and requests per minute
rate_analysis = df_clean.groupBy("ip") \
    .agg(
        count("*").alias("total_requests"),
        spark_min("time").alias("first_seen"),
        spark_max("time").alias("last_seen")
    ) \
    .withColumn(
        "duration_seconds",
        unix_timestamp("last_seen") - unix_timestamp("first_seen")
    ) \
    .withColumn(
        "requests_per_minute",
        spark_round(
            col("total_requests") / (col("duration_seconds") / 60 + 1), 2
        )
    ) \
    .orderBy("requests_per_minute", ascending=False)

print("=== TOP 20 IPs BY REQUEST RATE ===")
rate_analysis.show(20, truncate=False)

=== TOP 20 IPs BY REQUEST RATE ===
+---------------+--------------+-------------------+-------------------+----------------+-------------------+
|ip             |total_requests|first_seen         |last_seen          |duration_seconds|requests_per_minute|
+---------------+--------------+-------------------+-------------------+----------------+-------------------+
|91.185.146.55  |295           |2019-01-22 20:45:07|2019-01-22 20:45:24|17              |229.87             |
|151.242.164.92 |282           |2019-01-24 00:08:46|2019-01-24 00:09:01|15              |225.6              |
|77.237.185.216 |471           |2019-01-23 23:11:39|2019-01-23 23:12:45|66              |224.29             |
|185.179.222.246|294           |2019-01-25 18:07:26|2019-01-25 18:07:46|20              |220.5              |
|91.186.213.103 |301           |2019-01-24 23:42:36|2019-01-24 23:42:58|22              |220.24             |
|89.165.122.224 |319           |2019-01-22 11:30:59|2019-01-22 11:31:27|28           

In [ ]:
# This cell is just a summary of what we'll put in spark_app.py
# Run this to verify all transformations work as functions

def get_summary(df):
    from pyspark.sql.functions import count, col, round as spark_round
    total = df.count()
    breakdown = df.groupBy("type") \
        .agg(count("*").alias("count")) \
        .withColumn("percentage", spark_round((col("count") / total) * 100, 2)) \
        .orderBy("count", ascending=False)
    
    rows = [row.asDict() for row in breakdown.collect()]
    return {"total_requests": total, "type_breakdown": rows}

def get_top_attackers(df, limit=20):
    from pyspark.sql.functions import count, col, collect_set
    df_attacks = df.filter(col("type") != "benign")
    result = df_attacks.groupBy("ip") \
        .agg(
            count("*").alias("attack_count"),
            collect_set("type").alias("attack_types")
        ) \
        .orderBy("attack_count", ascending=False) \
        .limit(limit)
    
    rows = [row.asDict() for row in result.collect()]
    # Convert set to list for JSON serialization
    for row in rows:
        row["attack_types"] = list(row["attack_types"])
    return rows

# Test it
print(get_summary(df_clean))
print(get_top_attackers(df_clean, 5))